<a href="https://colab.research.google.com/github/Zihniii/Traveloka-Sentiment-Analysis/blob/main/Notebook/WEEK-5/05_Vector_Space_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📐 Week 5 — Vector Space Model (VSM)

---

### 🧠 Latar Belakang Konsep

**Mengapa kita perlu Vector Space Model?**

Komputer tidak bisa membaca teks secara langsung — mereka hanya memahami angka. VSM adalah cara untuk **mengubah dokumen teks menjadi vektor angka** sehingga kita bisa:
- Mengukur seberapa *mirip* dua dokumen
- Melakukan pencarian berbasis makna (semantic search)
- Mengelompokkan dokumen berdasarkan topik

**Intuisi utama:** Setiap dokumen direpresentasikan sebagai titik (vektor) di dalam ruang berdimensi-N, di mana N adalah jumlah kata unik dalam seluruh korpus. Dokumen yang membahas topik serupa akan berada *berdekatan* satu sama lain di ruang ini.

---

## Bagian 1: Dataset

Kita akan menggunakan korpus 5 dokumen bertema **perdagangan internasional** sebagai studi kasus. Dokumen 0 akan menjadi dokumen referensi (*query*) yang kita bandingkan dengan dokumen lainnya.

In [1]:
# ============================================================
# Import library yang dibutuhkan
# ============================================================
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# Dataset: 5 Dokumen Berita Perdagangan
# ============================================================
corpus = [
    "Analysts observed that repeated tariffs actions during the trade war created uncertainty in global trade and weakened trade growth.",  # Doc 0 (referensi)
    "Trump imposed new tariffs on Chinese goods during the trade war.",                                                                    # Doc 1
    "China responded to the trade conflict with counter tariffs.",                                                                         # Doc 2
    "The trade negotiations between Trump and China showed progress.",                                                                     # Doc 3
    "Tariffs impacted global trade and economic stability."                                                                                # Doc 4
]

print("📄 Korpus dokumen:")
for i, doc in enumerate(corpus):
    label = " ← dokumen referensi" if i == 0 else ""
    print(f"  Doc {i}: {doc}{label}")

📄 Korpus dokumen:
  Doc 0: Analysts observed that repeated tariffs actions during the trade war created uncertainty in global trade and weakened trade growth. ← dokumen referensi
  Doc 1: Trump imposed new tariffs on Chinese goods during the trade war.
  Doc 2: China responded to the trade conflict with counter tariffs.
  Doc 3: The trade negotiations between Trump and China showed progress.
  Doc 4: Tariffs impacted global trade and economic stability.


---
## Bagian 2: Term Frequency (TF) Matrix

Langkah pertama dalam VSM adalah menghitung **Term Frequency** — yaitu berapa kali setiap kata muncul di setiap dokumen.

**Format matriks TF:**
- Setiap **baris** = satu dokumen
- Setiap **kolom** = satu kata unik dari seluruh korpus
- Nilai sel = frekuensi kemunculan kata tersebut di dokumen tersebut

> ⚠️ **Catatan implementasi:** Di sini kita menggunakan `Counter` untuk membangun matriks TF secara manual tanpa preprocessing (lowercase, punctuation removal). Akibatnya, kata seperti `"tariffs"` dan `"tariffs."` dianggap berbeda. Ini adalah keterbatasan yang akan kita perbaiki di Bagian 3.

In [2]:
# ============================================================
# Membangun Term Frequency Matrix secara manual
# ============================================================
tf_matrix = pd.DataFrame(
    [Counter(doc.split()) for doc in corpus]
).fillna(0).astype(int)

print(f"Ukuran matriks TF: {tf_matrix.shape[0]} dokumen × {tf_matrix.shape[1]} kata unik")
print("\nMatriks Term Frequency (TF) — hanya kolom yang relevan:")

# Tampilkan hanya kolom yang paling informatif
key_cols = ['trade', 'tariffs', 'Tariffs', 'global', 'war', 'China', 'Trump']
key_cols_exist = [c for c in key_cols if c in tf_matrix.columns]
print(tf_matrix[key_cols_exist])

Ukuran matriks TF: 5 dokumen × 40 kata unik

Matriks Term Frequency (TF) — hanya kolom yang relevan:
   trade  tariffs  Tariffs  global  war  China  Trump
0      3        1        0       1    1      0      0
1      1        1        0       0    0      0      1
2      1        0        0       0    0      1      0
3      1        0        0       0    0      1      1
4      1        0        1       1    0      0      0


---
## Bagian 3: Representasi Vektor dengan Vocabulary Terkontrol

Daripada menggunakan semua kata (yang menghasilkan matriks sangat lebar dan sparse), kita bisa **membatasi vocabulary** hanya pada kata-kata yang paling penting.

Di sini kita fokus pada dua kata kunci: `"trade"` dan `"tariffs"` — dua term yang paling sering muncul di topik perdagangan internasional.

Dengan dua dimensi ini, setiap dokumen bisa digambarkan sebagai titik di bidang 2D — sangat mudah untuk divisualisasikan!

In [3]:
# ============================================================
# Representasi vektor dengan vocabulary terbatas
# Menggunakan CountVectorizer dari scikit-learn
# ============================================================
vocabulary = ['trade', 'tariffs']
vectorizer = CountVectorizer(vocabulary=vocabulary)
X = vectorizer.fit_transform(corpus)

# Buat DataFrame untuk tampilan yang lebih rapi
df_vectors = pd.DataFrame(
    X.toarray(),
    columns=vocabulary,
    index=[f'Doc {i}' for i in range(len(corpus))]
)

print("Vektor setiap dokumen (dimensi: trade, tariffs):")
print(df_vectors)
print("\nInterpretasi:")
for idx, row in df_vectors.iterrows():
    print(f"  {idx}: vektor = ({row['trade']}, {row['tariffs']}) — {corpus[int(idx.split()[1])][:50]}...")

Vektor setiap dokumen (dimensi: trade, tariffs):
       trade  tariffs
Doc 0      3        1
Doc 1      1        1
Doc 2      1        1
Doc 3      1        0
Doc 4      1        1

Interpretasi:
  Doc 0: vektor = (3, 1) — Analysts observed that repeated tariffs actions du...
  Doc 1: vektor = (1, 1) — Trump imposed new tariffs on Chinese goods during ...
  Doc 2: vektor = (1, 1) — China responded to the trade conflict with counter...
  Doc 3: vektor = (1, 0) — The trade negotiations between Trump and China sho...
  Doc 4: vektor = (1, 1) — Tariffs impacted global trade and economic stabili...


---
## Bagian 4: Cosine Similarity

### Mengapa menggunakan Cosine Similarity, bukan jarak biasa?

**Euclidean Distance** mengukur *jarak absolut* antar dua titik. Masalahnya: dokumen yang lebih panjang secara alami akan memiliki nilai frekuensi lebih tinggi, sehingga terlihat "jauh" dari dokumen pendek meski membahas topik yang sama.

**Cosine Similarity** mengukur *sudut* antara dua vektor, bukan jaraknya. Dokumen panjang dan dokumen pendek yang membahas topik sama akan memiliki *arah* yang serupa → cosine similarity tinggi.

$$\text{cos}(\theta) = \frac{\vec{A} \cdot \vec{B}}{|\vec{A}| \cdot |\vec{B}|}$$

| Nilai Cosine Similarity | Interpretasi |
|---|---|
| 1.0 | Identik secara topik |
| 0.7 – 0.99 | Sangat mirip |
| 0.4 – 0.69 | Agak mirip |
| 0.0 | Sama sekali berbeda |

In [5]:
# ============================================================
# Hitung Cosine Similarity: Doc 0 vs semua dokumen
# ============================================================
cos_sim_to_doc0 = cosine_similarity(X[0:1], X).flatten()

# Gabungkan hasil ke dalam DataFrame
df_result = df_vectors.copy()
df_result['Cosine Sim vs Doc 0'] = cos_sim_to_doc0
df_result['Ranking'] = df_result['Cosine Sim vs Doc 0'].rank(ascending=False).astype(int)

print("Cosine Similarity setiap dokumen terhadap Doc 0 (dokumen referensi):")
print(df_result.sort_values('Cosine Sim vs Doc 0', ascending=False))

Cosine Similarity setiap dokumen terhadap Doc 0 (dokumen referensi):
       trade  tariffs  Cosine Sim vs Doc 0  Ranking
Doc 0      3        1             1.000000        1
Doc 3      1        0             0.948683        2
Doc 1      1        1             0.894427        4
Doc 2      1        1             0.894427        4
Doc 4      1        1             0.894427        4


---
## Bagian 5: Full Cosine Similarity Matrix

Selain membandingkan terhadap satu dokumen referensi, kita bisa menghitung **matriks kemiripan penuh** — yaitu perbandingan *semua pasangan* dokumen sekaligus.

Matriks ini bersifat **simetris**: kemiripan Doc A terhadap Doc B selalu sama dengan Doc B terhadap Doc A. Nilai diagonal selalu 1.0 karena setiap dokumen identik dengan dirinya sendiri.

In [7]:
# ============================================================
# Hitung Full Cosine Similarity Matrix (semua pasangan)
# ============================================================
full_cos_sim = cosine_similarity(X)

doc_labels = [f'Doc {i}' for i in range(len(corpus))]
cos_sim_df = pd.DataFrame(full_cos_sim, index=doc_labels, columns=doc_labels)

print("Matriks Cosine Similarity Penuh (semua pasangan dokumen):")
print(cos_sim_df.round(4))

Matriks Cosine Similarity Penuh (semua pasangan dokumen):
        Doc 0   Doc 1   Doc 2   Doc 3   Doc 4
Doc 0  1.0000  0.8944  0.8944  0.9487  0.8944
Doc 1  0.8944  1.0000  1.0000  0.7071  1.0000
Doc 2  0.8944  1.0000  1.0000  0.7071  1.0000
Doc 3  0.9487  0.7071  0.7071  1.0000  0.7071
Doc 4  0.8944  1.0000  1.0000  0.7071  1.0000


---
## Bagian 6: Cosine Similarity vs Euclidean Distance

Mari kita buktikan secara empiris mengapa Cosine Similarity lebih robust daripada Euclidean Distance untuk teks.

Kita akan bandingkan Doc 0 (dokumen panjang) dengan Doc 1 dan Doc 2 menggunakan **kedua metrik**, lalu lihat apakah hasilnya konsisten.

In [9]:
# ============================================================
# Perbandingan: Cosine Similarity vs Euclidean Distance
# Menggunakan Full TF Matrix (semua kata)
# ============================================================

def vector_length(v):
    """Menghitung panjang (magnitude) sebuah vektor."""
    return np.sqrt((v ** 2).sum())

def cosine_distance(v, w):
    """Cosine Distance = 1 - Cosine Similarity. Semakin kecil = semakin mirip."""
    return 1 - (v * w).sum() / (vector_length(v) * vector_length(w))

def euclidean_distance(v, w):
    """Jarak Euclidean standar."""
    return np.sqrt(((v - w) ** 2).sum())

# Gunakan full TF matrix
tf_matrix_int = pd.DataFrame(
    [Counter(doc.split()) for doc in corpus]
).fillna(0).astype(int)

# Hitung jarak antara Doc 0 dan Doc 1, serta Doc 0 dan Doc 2
pairs = [(0, 1), (0, 2), (0, 3), (0, 4)]

print("Perbandingan Cosine Distance vs Euclidean Distance:")
print(f"{'Pasangan':<15} {'Cosine Dist':<18} {'Euclidean Dist':<18} {'Ukuran Doc A':<15} {'Ukuran Doc B'}")
print("-" * 80)

for a, b in pairs:
    va = tf_matrix_int.loc[a]
    vb = tf_matrix_int.loc[b]
    cd = cosine_distance(va, vb)
    ed = euclidean_distance(va, vb)
    len_a = len(corpus[a].split())
    len_b = len(corpus[b].split())
    print(f"Doc {a} vs Doc {b}     {cd:<18.4f} {ed:<18.4f} {len_a:<15} {len_b}")

Perbandingan Cosine Distance vs Euclidean Distance:
Pasangan        Cosine Dist        Euclidean Dist     Ukuran Doc A    Ukuran Doc B
--------------------------------------------------------------------------------
Doc 0 vs Doc 1     0.6382             4.8990             19              11
Doc 0 vs Doc 2     0.7333             5.0990             19              9
Doc 0 vs Doc 3     0.7333             5.0990             19              9
Doc 0 vs Doc 4     0.6220             4.6904             19              7


---
## 📌 Ringkasan & Kesimpulan

| Konsep | Penjelasan Singkat |
|---|---|
| **Vector Space Model** | Representasi dokumen sebagai vektor dalam ruang multi-dimensi |
| **Term Frequency (TF)** | Jumlah kemunculan kata dalam satu dokumen |
| **Cosine Similarity** | Mengukur kemiripan sudut antar vektor, tidak terpengaruh panjang dokumen |
| **Euclidean Distance** | Mengukur jarak absolut, sensitif terhadap panjang dokumen |
| **TF-IDF** | Pembobotan yang lebih cerdas: memprioritaskan kata unik & informatif |

### Aplikasi di Dunia Nyata
- 🔍 **Mesin pencari** (Google, Bing): mencocokkan query dengan dokumen relevan
- 📺 **Sistem rekomendasi**: menyarankan konten serupa berdasarkan teks
- 📧 **Filter spam**: mengklasifikasikan email berdasarkan kata-kata yang digunakan
- 📰 **Clustering berita**: mengelompokkan artikel berdasarkan topik

---
*Notebook ini adalah bagian dari mata kuliah Pemrosesan Bahasa Alami (PBA) — Week 5*